# Homework 4 Solution

This notebook shows one possible solution for the multivariable linear regression homework based on the China panel dataset.

For readability, this solution creates a few short English aliases for Chinese variable names while keeping the original source columns intact.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_04"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q openpyxl seaborn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel(
    DATA_DIR / "china" / "china_panel_data.xlsx",
    sheet_name=0,
)

province_map = pd.read_csv(DATA_DIR / "china" / "province_name_mapping.csv")

df = df.merge(province_map, left_on="地区", right_on="region_cn", how="left")
df.head()

In [ ]:
df.columns

# Q1. Select regression variables and produce descriptive statistics.

A common choice is to use GDP per capita as the dependent variable and variables such as tertiary-industry value added, exports, e-commerce activity, and R&D expenditure as explanatory variables.

When the standard deviation is very large, a log transformation is often useful to reduce scale differences and potential heteroskedasticity.

In [ ]:
df["year"] = df["年份"]
df["gdp_per_capita"] = df["国内生产总值（当年价）（亿元）"] / df["总人口（万人）"]
df["lngdpc"] = np.log(df["gdp_per_capita"])
df["urbanization_rate"] = df["城镇人口（万人）"] / df["总人口（万人）"] * 100
df["lnrd_pc"] = np.log(df["R&D经费（万元）"] / df["总人口（万人）"])
df["digital_finance_index"] = df["数字普惠金融指数"]
df["ecommerce_share_pct"] = df["有电子商务交易活动企业占总企业数比重（%）"]
df["lntrade_pc"] = np.log(
    (df["货物出口金额（万美元）"] + df["货物进口金额（万美元）"])
    / df["总人口（万人）"]
)

df[[
    "gdp_per_capita",
    "lngdpc",
    "urbanization_rate",
    "lnrd_pc",
    "digital_finance_index",
    "ecommerce_share_pct",
    "lntrade_pc",
]].describe().T



# Q2. Visualize selected variables. Different regions should be visually distinguishable.

In [ ]:
sns.scatterplot(
    data=df,
    x="urbanization_rate",
    y="lngdpc",
    hue="region_en",
    alpha=0.7,
)
plt.xlabel("Urbanization rate (%)")
plt.ylabel("Log GDP per capita")
plt.title("Urbanization and GDP per capita")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "scatter_urbanization_lngdpc.png", bbox_inches="tight")



In [ ]:
sns.scatterplot(
    data=df,
    x="digital_finance_index",
    y="lngdpc",
    hue="year",
    palette="viridis",
    alpha=0.75,
)
plt.xlabel("Digital finance index")
plt.ylabel("Log GDP per capita")
plt.title("Digital finance and GDP per capita")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "scatter_digital_finance_lngdpc.png", bbox_inches="tight")



In [ ]:
sns.scatterplot(data=df, x="lnrd_pc", y="lngdpc", hue="region_en")
plt.xlabel("Log R&D expenditure per capita")
plt.ylabel("Log GDP per capita")
plt.title("Innovation input and GDP per capita")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "scatter_lnrdpc_lngdpc.png", bbox_inches="tight")



In [ ]:
df["area_group"] = "Non-coastal region"

df.loc[df["地区"].isin(["上海", "江苏", "浙江", "福建", "广东"]), "area_group"] = "Southeastern coastal region"

df["area_group"].unique()

In [ ]:
sns.lineplot(
    data=df,
    x="year",
    y="digital_finance_index",
    hue="area_group",
    estimator="mean",
    errorbar=None,
)
plt.ylabel("Average digital finance index")
plt.title("Digital finance by area group")
plt.legend(loc="best")
plt.savefig(MODULE_OUTPUT_DIR / "line_digital_finance_by_area_group.png", bbox_inches="tight")



# Q3. Run OLS regressions using the selected variables.

In [ ]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

# Define the dependent variable.
y = df[["lngdpc"]]

# Define the independent variables.
X1 = df[["ecommerce_share_pct"]]
X2 = df[["ecommerce_share_pct", "digital_finance_index"]]
X3 = df[["ecommerce_share_pct", "digital_finance_index", "urbanization_rate"]]
X4 = df[["ecommerce_share_pct", "digital_finance_index", "urbanization_rate", "lnrd_pc"]]

# Add constant terms.
X1 = sm.add_constant(X1)
X2 = sm.add_constant(X2)
X3 = sm.add_constant(X3)
X4 = sm.add_constant(X4)

# Fit OLS models.
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
model4 = sm.OLS(y, X4).fit()

# Create a summary table with significance stars.
reg = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=["Model 1", "Model 2", "Model 3", "Model 4"],
    float_format="%0.3f",
    info_dict={"N": lambda result: result.nobs, "R2": lambda result: f"{result.rsquared:.3f}"},
)

reg_df = reg.tables[0].reset_index(drop=False)
reg_df.to_excel(MODULE_OUTPUT_DIR / "Homework4_regression_results.xlsx", index=False)

reg_df

